# RescueNet PSPNet – Kaggle'da Eğitim (3 sınıf, 897×897)

Bu notebook Kaggle GPU (2× T4) ortamında RescueNet için PSPNet eğitimi yapar.
- **Sınıflar**: Background, Building Light, Building Heavy (3 sınıf)
- **Çözünürlük**: 897×897
- **Resume**: Kesinti sonrası en son checkpoint ile devam edilebilir.

**Gerekli**: Kaggle'da "Add Data" ile **yaroslavchyrko/rescuenet** veri setini ekleyin.

## 0. Repo güncelleme (isteğe bağlı)

Güncelleme yaptıktan sonra bu hücreyi çalıştırın: repo klonlanır veya zaten varsa `git pull` ile güncellenir. **REPO_URL**'i kendi GitHub reponuzla değiştirin.

In [ ]:
# Kendi reponuzun URL'si (ör: https://github.com/KULLANICI_ADI/RescueNetModelTraining.git)
REPO_URL = "https://github.com/nickotyn46/RescueNetModelTraining"
REPO_DIR = "RescueNetModelTraining"

import os
import subprocess

# Repo kökü (notebook Segmentation-Experiments içindeyse üst dizin)
repo_root = os.getcwd() if os.path.isdir(".git") else (os.path.join(os.getcwd(), "..") if os.path.isdir(os.path.join(os.getcwd(), "..", ".git")) else None)
if repo_root:
    print("Repo mevcut, güncelleniyor (git pull)...")
    out = subprocess.run(["git", "pull"], cwd=repo_root, capture_output=True, text=True)
    print(out.stdout or out.stderr or "OK")
elif not os.path.isfile("train.py") or not os.path.isdir("Segmentation-Experiments"):
    print("Repo klonlanıyor...")
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL], check=True)
    seg_exp = os.path.join(REPO_DIR, "Segmentation-Experiments")
    if os.path.isdir(seg_exp):
        os.chdir(seg_exp)
        print("Dizin:", os.getcwd())
    elif os.path.isdir(REPO_DIR):
        os.chdir(REPO_DIR)
        print("Dizin:", os.getcwd())
else:
    print("train.py veya Segmentation-Experiments zaten mevcut; clone atlandı. Güncellemek için önce bu dizinde git init/pull yapın.")

## 1. Ortam ve path kontrolü

### 1a. Önce dizini öğren (bu hücreyi çalıştır, çıktıdaki path'i kopyala)

Aşağıdaki hücre `/kaggle/input` altındaki tüm dizinleri listeler. **train** ve **train-org-img** gördüğün dizinin path'ini kopyala; bir sonraki hücrede `DATA_ROOT = "BURAYA_YAPIŞTIR"` yazacaksın.

In [ ]:
import os

KAGGLE_INPUT = "/kaggle/input"
# /kaggle/input altını 3 seviye derinliğe kadar listele
for root, dirs, files in os.walk(KAGGLE_INPUT):
    depth = root.count(os.sep) - KAGGLE_INPUT.count(os.sep)
    if depth > 3:
        continue
    indent = "  " * depth
    print(f"{indent}{root}/")
    for d in dirs[:15]:  # her dizinde max 15 alt dizin
        print(f"{indent}  [{d}]")
    if not dirs and files:
        print(f"{indent}  (dosyalar: {len(files)} adet)")
    print()

In [ ]:
import os
import sys
import glob

KAGGLE_INPUT = "/kaggle/input"
KAGGLE_WORKING = "/kaggle/working"

# --- Dizin listesi hücresinden gördüğün path'i buraya yapıştır (örn: /kaggle/input/datasets/RescueNet) ---
MANUAL_DATA_ROOT = None  # None bırakırsan otomatik aranır; bulunamazsa yukarıdaki hücreden path kopyala
# ---------------------------------------------------------------------------------------------------------

def find_rescuenet_root():
    candidates = [
        os.path.join(KAGGLE_INPUT, "datasets", "yaroslavchyrko", "rescuenet", "RescueNet"),
        os.path.join(KAGGLE_INPUT, "rescuenet"),
        os.path.join(KAGGLE_INPUT, "rescuenet", "RescueNet"),
        os.path.join(KAGGLE_INPUT, "datasets", "rescuenet"),
        os.path.join(KAGGLE_INPUT, "datasets", "RescueNet"),
    ]
    for p in candidates:
        if os.path.isdir(p):
            train_dir = os.path.join(p, "train", "train-org-img")
            if os.path.isdir(train_dir):
                return p
    # datasets/ altındaki her alt dizinde train/train-org-img ara
    datasets_dir = os.path.join(KAGGLE_INPUT, "datasets")
    if os.path.isdir(datasets_dir):
        for name in os.listdir(datasets_dir):
            p = os.path.join(datasets_dir, name)
            if os.path.isdir(p) and os.path.isdir(os.path.join(p, "train", "train-org-img")):
                return p
    return None

DATA_ROOT = MANUAL_DATA_ROOT if MANUAL_DATA_ROOT else find_rescuenet_root()
if DATA_ROOT is None:
    print("UYARI: RescueNet veri dizini bulunamadı. Add Data -> yaroslavchyrko/rescuenet ekleyin.")
    print("Mevcut input dizinleri:", os.listdir(KAGGLE_INPUT) if os.path.isdir(KAGGLE_INPUT) else "N/A")
    if os.path.isdir(os.path.join(KAGGLE_INPUT, "datasets")):
        print("datasets/ içeriği:", os.listdir(os.path.join(KAGGLE_INPUT, "datasets")))
else:
    print("DATA_ROOT:", DATA_ROOT)

# Çalışma dizini: bu notebook Segmentation-Experiments içinde olmalı
CODE_DIR = os.path.dirname(os.path.abspath("train.py")) if os.path.isfile("train.py") else os.getcwd()
if os.path.basename(CODE_DIR) != "Segmentation-Experiments" and os.path.isdir("Segmentation-Experiments"):
    CODE_DIR = os.path.join(os.getcwd(), "Segmentation-Experiments")
os.chdir(CODE_DIR)
sys.path.insert(0, CODE_DIR)
print("Çalışma dizini:", os.getcwd())

## 2. Bağımlılıklar

In [ ]:
!pip install -q tensorboardX PyYAML gdown

## 3. Resume path belirleme ve config override

In [ ]:
import yaml

CONFIG_PATH = "config/rescuenet-pspnet101.yaml"
SAVE_PATH = os.path.join(KAGGLE_WORKING, "exp", "pspnet101", "model") if os.path.isdir(KAGGLE_WORKING) else "exp/RescueNet/pspnet101/model"

# Epoch sayısı (planlanan: 100; isterseniz düşürüp hızlı deneme yapabilirsiniz)
EPOCHS = 100

# En son checkpoint'i bul (önceki run çıktısı veya bu oturum)
def find_latest_checkpoint():
    patterns = []
    if os.path.isdir(KAGGLE_INPUT):
        for name in os.listdir(KAGGLE_INPUT):
            d = os.path.join(KAGGLE_INPUT, name, "exp", "pspnet101", "model")
            if os.path.isdir(d):
                patterns.append(os.path.join(d, "train_epoch_*.pth"))
    if os.path.isdir(SAVE_PATH):
        patterns.append(os.path.join(SAVE_PATH, "train_epoch_*.pth"))
    latest = None
    latest_epoch = -1
    for pat in patterns:
        for path in glob.glob(pat):
            try:
                epoch = int(path.split("train_epoch_")[1].split(".pth")[0])
                if epoch > latest_epoch:
                    latest_epoch = epoch
                    latest = path
            except (IndexError, ValueError):
                pass
    return latest

RESUME_PATH = find_latest_checkpoint()
if RESUME_PATH:
    print("Resume checkpoint:", RESUME_PATH)
else:
    print("Resume yok, sıfırdan başlanacak.")

# Config dosyasını oku ve Kaggle için override et
with open(CONFIG_PATH, "r") as f:
    cfg = yaml.safe_load(f)

if DATA_ROOT:
    cfg["DATA"]["data_root"] = DATA_ROOT
cfg["TRAIN"]["save_path"] = SAVE_PATH
cfg["DATA"]["classes"] = 3
cfg["TRAIN"]["train_h"] = 897
cfg["TRAIN"]["train_w"] = 897
cfg["TEST"]["test_h"] = 897
cfg["TEST"]["test_w"] = 897
cfg["TRAIN"]["resume"] = RESUME_PATH or ""
# 2x T4 GPU kullanımı (DataParallel otomatik kullanılır)
cfg["TRAIN"]["train_gpu"] = [0, 1]
cfg["TRAIN"]["epochs"] = EPOCHS
# Her epoch sonunda validation mIoU/loss da yazdırılsın
cfg["TRAIN"]["evaluate"] = True

# Override edilmiş config'i geçici yaz (train.py YAML'dan okuyacak)
KAGGLE_CONFIG = "config/rescuenet-pspnet101-kaggle.yaml"
with open(KAGGLE_CONFIG, "w") as f:
    yaml.dump(cfg, f, default_flow_style=False, allow_unicode=True)

print("Kaggle config yazıldı:", KAGGLE_CONFIG)
print("data_root:", cfg["DATA"]["data_root"])
print("save_path:", cfg["TRAIN"]["save_path"])
print("train_gpu (2x T4):", cfg["TRAIN"]["train_gpu"])
print("epochs:", cfg["TRAIN"]["epochs"])

## 4. Eğitimi çalıştır

- **2x T4**: `train_gpu: [0, 1]` ile her iki GPU da `DataParallel` ile kullanılır; batch otomatik bölünür.
- **Epoch sonuçları**: Her epoch bitiminde aşağıdaki hücre çıktısında **Train result** (mIoU, mAcc, allAcc) ve **Val result** (evaluate=True olduğu için) görünür. Checkpoint'ler `save_freq` (1) ile her epoch sonunda kaydedilir.
- **Epoch sayısı**: Varsayılan **100**; yukarıdaki hücrede `EPOCHS` ile değiştirebilirsiniz.

In [ ]:
# Eğitimi başlat (checkpoint'ler save_path altına yazılır)
!python train.py --config config/rescuenet-pspnet101-kaggle.yaml

### Kesintiden sonra devam
- Checkpoint'ler **save_path** altına (Kaggle'da `/kaggle/working/exp/pspnet101/model/`) kaydedilir.
- **Save Version** ile çıktıyı kaydedin. Sonraki çalıştırmada bu çıktıyı **Add Data** ile ekleyin; notebook en son checkpoint'i bulup `resume` ile devam eder.

In [ ]:
# Opsiyonel: Checkpoint listesi
import glob
if os.path.isdir(SAVE_PATH):
    ckpts = sorted(glob.glob(os.path.join(SAVE_PATH, "train_epoch_*.pth")))
    print("Kaydedilen checkpoint'ler:", ckpts[-5:] if len(ckpts) > 5 else ckpts)